In [ ]:
# === Cell 1: Imports & Config & Utils (use pre-generated networks) ===
import os
import re
import glob
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings("ignore")

# -----------------------------
# Config
# -----------------------------
NETWORK_DIR        = "network"          # directory of the C{cid}_network.txt files from the network export
OUTPUT_DIR         = "edge_cluster"   # output directory of this analysis
# cluster order along the lineage: trunk/branch 1 (4-8-7-3-2), then branch 2 (1-5)
DESIRED_ORDER      = [4, 8, 7, 3, 2, 1, 5]
CLUSTERS: Optional[List[int]] = None    # cluster IDs to use; None parses them from the C{cid}_network.txt file names

SELECTION_MODE     = "all"              # "all" / "topk" / "threshold" / "both"
TOP_N_PER_CLUSTER  = 10000               # top-N edges per cluster (used only when the mode contains "topk")
SCORE_THRESHOLD    = 0.90               # score threshold (used only when the mode contains "threshold")

N_CLUSTERS         =12                 # number of edge clusters (KMeans)
RANDOM_STATE       = 5465                 # random seed
HEATMAP_CMAP       = "OrRd"             # heatmap colormap
SAVE_FIG           = False              # save figures to OUTPUT_DIR

TOP_TF_PER_CLUSTER = 20                 # top TFs exported per cluster (by degree)
TOP_TG_PER_CLUSTER = 20                 # top TGs exported per cluster (by degree)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# -----------------------------
# Utils
# -----------------------------
def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)


def discover_clusters(network_dir: str) -> List[int]:
    """
    Parse the cluster IDs from the C{cid}_network.txt file names in network_dir.
    Accepts e.g. C4_network.txt, C4_network.tsv and C4_network.txt.gz.
    """
    paths = glob.glob(os.path.join(network_dir, "C*_network.*"))
    cids = []
    for p in paths:
        m = re.search(r"C(\d+)_network\.", os.path.basename(p))
        if m:
            cids.append(int(m.group(1)))
    return sorted(set(cids))


def load_network_for_cluster(cid: int, network_dir: str) -> pd.DataFrame:
    """
    Read one cluster's network file; return a DataFrame with at least the columns [TF, TG, Score].
    The file is tab-separated with a header.
    """
    # candidate file names (C{cid}_network.txt and variants)
    candidates = [
        os.path.join(network_dir, f"C{cid}_network.txt"),
        os.path.join(network_dir, f"C{cid}_network.tsv"),
    ]
    path = None
    for p in candidates:
        if os.path.exists(p):
            path = p
            break
    assert path is not None, f"Network file for C{cid} not found in {network_dir}"

    df = pd.read_csv(path, sep="\t")

    # normalize the column-name case
    cols = {c.lower(): c for c in df.columns}
    # expects tf / tg / score columns
    tf_col = cols.get("tf", None)
    tg_col = cols.get("tg", None)
    sc_col = cols.get("score", None)
    assert tf_col and tg_col and sc_col, f"{path} must contain TF/TG/Score columns."

    df = df[[tf_col, tg_col, sc_col]].copy()
    df.columns = ["TF", "TG", "Score"]
    return df


def select_edges_df(df: pd.DataFrame,
                    mode: str = "all",
                    top_n: int = 3000,
                    thr: float = 0.9) -> pd.DataFrame:
    """
    Filter one cluster's network (columns [TF, TG, Score]):
      - "all": keep every edge
      - "topk": keep the top_n edges by Score
      - "threshold": keep edges with Score >= thr
      - "both": topk, then intersect with threshold
    Return the filtered DataFrame (TF/TG/Score).
    """
    if df.empty:
        return df

    if mode == "all":
        return df.sort_values("Score", ascending=False).reset_index(drop=True)

    scores = df["Score"].values
    idx_all = np.arange(len(df))

    idx_topk = idx_thr = None

    if mode in ("topk", "both"):
        k = min(top_n, len(df))
        part = np.argpartition(-scores, kth=k-1)[:k]
        part = part[np.argsort(-scores[part])]
        idx_topk = part

    if mode in ("threshold", "both"):
        idx_thr = idx_all[scores >= thr]

    if mode == "topk":
        chosen = idx_topk
    elif mode == "threshold":
        chosen = idx_thr
    else:  # both
        chosen = np.intersect1d(idx_topk, idx_thr, assume_unique=False)

    if chosen is None or len(chosen) == 0:
        return df.iloc[[]].copy()

    return df.iloc[chosen].sort_values("Score", ascending=False).reset_index(drop=True)


def build_tf_tg_indices(df_edges: pd.DataFrame) -> Tuple[Dict[str, int], Dict[str, int]]:
    """
    Build integer indices for all TF/TG names.
    """
    tf_names = sorted(df_edges["tf_name"].unique().tolist())
    tg_names = sorted(df_edges["tg_name"].unique().tolist())
    tf2idx = {name: i for i, name in enumerate(tf_names)}
    tg2idx = {name: i for i, name in enumerate(tg_names)}
    return tf2idx, tg2idx


def plot_heatmap_with_ec_bar(mat: np.ndarray,
                             xticklabels: List[str],
                             yticklabels: List[str],
                             title: str,
                             fname: str,
                             ec_counts: np.ndarray):
    """
    Draw the edge_cluster × cell_cluster heatmap with a bar chart on the right
    and an EC label column on the left.
    """
    n_ec = mat.shape[0]
    assert n_ec == len(yticklabels) == len(ec_counts)

    # canvas size for 20/24 pt fonts
    fig_w = max(12, 1.2 * len(xticklabels)) + 5.0
    fig_h = max(8, 0.8 * n_ec)
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = GridSpec(nrows=1, ncols=2, figure=fig,
                  width_ratios=[20, 9], wspace=0.28)

    # heatmap
    ax_hm = fig.add_subplot(gs[0, 0])
    im = ax_hm.imshow(mat, aspect="auto", cmap=HEATMAP_CMAP)
    ax_hm.set_title(title, pad=20) # pad keeps the 24 pt title clear of the plot
    ax_hm.set_xticks(np.arange(len(xticklabels)))
    ax_hm.set_xticklabels(xticklabels, rotation=45, ha="right")
    ax_hm.set_yticks(np.arange(n_ec))
    ax_hm.set_yticklabels(yticklabels)

    # colorbar
    divider = make_axes_locatable(ax_hm)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Proportion" if "prop" in fname else "Count")

    # EC labels on the left
    ax_lbl = divider.append_axes("left", size="5%", pad=0.05)
    ax_lbl.set_ylim(-0.5, n_ec - 0.5)
    ax_lbl.set_xlim(0.0, 1.0)
    ax_lbl.invert_yaxis()
    ax_lbl.axis("off")
    for y in range(n_ec):
        # font size 20
        ax_lbl.text(0.995, y, f"{yticklabels[y]}",
                    va="center", ha="right", fontsize=20)

    # bar chart on the right
    ax_bar = fig.add_subplot(gs[0, 1], sharey=ax_hm)
    y_pos = np.arange(n_ec)
    ax_bar.barh(y_pos, ec_counts, align="center")
    ax_bar.set_xlabel("edge count")
    ax_bar.set_yticks([])
    ax_bar.grid(axis="x", linestyle="--", alpha=0.3)
    ax_bar.set_xlim(0, max(1.0, ec_counts.max()) * 1.08)

    # margins so that large fonts are not clipped
    fig.subplots_adjust(left=0.12, right=0.95, top=0.88, bottom=0.18)
    if SAVE_FIG:
        ensure_dir(OUTPUT_DIR)
        fig.savefig(os.path.join(OUTPUT_DIR, fname), dpi=200)
    plt.show()

In [ ]:
# === Cell 2: Use pre-generated networks to cluster edges & visualize ===

# === Cell 2: Use pre-generated networks to cluster edges & visualize ===

# 0) find the cluster IDs
if CLUSTERS is None:
    CLUSTERS = discover_clusters(NETWORK_DIR)
assert CLUSTERS, f"No C{{cid}}_network.* files found in {NETWORK_DIR}"

# ================= order CLUSTERS along the developmental trajectory =================
if 'DESIRED_ORDER' in globals() and DESIRED_ORDER:
    # follow the trajectory order and drop clusters that were not found
    ordered_clusters = [c for c in DESIRED_ORDER if c in CLUSTERS]
    # append clusters found but not in the order, so that no data is dropped
    remaining = [c for c in CLUSTERS if c not in ordered_clusters]
    CLUSTERS = ordered_clusters + remaining
# =================================================================

print("Using clusters ordered by trajectory:", CLUSTERS)

# 1) read each cluster's network, optionally filter, and collect into df_edges
records = []

for cid in CLUSTERS:
    df_net = load_network_for_cluster(cid, NETWORK_DIR)
    df_sel = select_edges_df(df_net,
                             mode=SELECTION_MODE,
                             top_n=TOP_N_PER_CLUSTER,
                             thr=SCORE_THRESHOLD)

    if df_sel.empty:
        print(f"[Warn] C{cid}: no edges after selection, skipped.")
        continue

    for _, r in df_sel.iterrows():
        records.append({
            "cell_cluster": cid,
            "tf_name": str(r["TF"]),
            "tg_name": str(r["TG"]),
            "score": float(r["Score"])
        })

df_edges = pd.DataFrame(records)
if df_edges.empty:
    raise RuntimeError("No edges collected from network/; "
                       "check NETWORK_DIR / SELECTION_MODE / thresholds.")

# 2) global TF/TG indices (tf_idx / tg_idx)
tf2idx, tg2idx = build_tf_tg_indices(df_edges)
tf_idx2name = {i: n for n, i in tf2idx.items()}
tg_idx2name = {i: n for n, i in tg2idx.items()}

df_edges["tf_idx"] = df_edges["tf_name"].map(tf2idx).astype(int)
df_edges["tg_idx"] = df_edges["tg_name"].map(tg2idx).astype(int)

# 3) export each cluster's edges (by descending score)
df_edges_sorted = df_edges.sort_values(
    ["cell_cluster", "score"], ascending=[True, False]
).reset_index(drop=True)

df_edges_sorted.to_csv(
    os.path.join(OUTPUT_DIR, f"edges_selected_from_network_{SELECTION_MODE}.csv"),
    index=False
)

for cid, sub in df_edges_sorted.groupby("cell_cluster"):
    sub.to_csv(
        os.path.join(OUTPUT_DIR, f"edges_C{cid}_{SELECTION_MODE}.csv"),
        index=False
    )

# 4) top TFs/TGs per cluster (by degree)
def export_top_nodes_by_degree(df_edges: pd.DataFrame,
                               top_tf: int = 20,
                               top_tg: int = 20):
    tf_rows, tg_rows = [], []

    for cid, sub in df_edges.groupby("cell_cluster"):
        tf_deg = (sub.groupby(["tf_idx", "tf_name"])
                     .size()
                     .sort_values(ascending=False))
        tg_deg = (sub.groupby(["tg_idx", "tg_name"])
                     .size()
                     .sort_values(ascending=False))

        tf_top = tf_deg.head(top_tf)
        tg_top = tg_deg.head(top_tg)

        tf_rows.append(pd.DataFrame({
            "cell_cluster": cid,
            "tf_idx": [i for (i, _) in tf_top.index],
            "tf_name": [n for (_, n) in tf_top.index],
            "degree": tf_top.values
        }))

        tg_rows.append(pd.DataFrame({
            "cell_cluster": cid,
            "tg_idx": [i for (i, _) in tg_top.index],
            "tg_name": [n for (_, n) in tg_top.index],
            "degree": tg_top.values
        }))

    dft = pd.concat(tf_rows, ignore_index=True)
    dgg = pd.concat(tg_rows, ignore_index=True)

    dft.to_csv(os.path.join(OUTPUT_DIR,
                            f"top_TF_by_degree_from_network_{SELECTION_MODE}.csv"),
               index=False)
    dgg.to_csv(os.path.join(OUTPUT_DIR,
                            f"top_TG_by_degree_from_network_{SELECTION_MODE}.csv"),
               index=False)
    return dft, dgg

dft, dgg = export_top_nodes_by_degree(
    df_edges_sorted,
    TOP_TF_PER_CLUSTER,
    TOP_TG_PER_CLUSTER
)

# 5) edge × cluster presence matrix X (present = 1)
edge_tuples = pd.MultiIndex.from_frame(
    df_edges_sorted[["tf_idx", "tg_idx"]]
).unique().tolist()
edge2row = {e: i for i, e in enumerate(edge_tuples)}
cid2col = {cid: j for j, cid in enumerate(CLUSTERS)}

X = np.zeros((len(edge_tuples), len(CLUSTERS)), dtype=np.float32)

for _, r in df_edges_sorted.iterrows():
    row = edge2row[(int(r.tf_idx), int(r.tg_idx))]
    col = cid2col[int(r.cell_cluster)]
    X[row, col] = 1.0

# 6) edge clustering (KMeans on standardized signatures)
if len(edge_tuples) > 1 and N_CLUSTERS > 1:
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xz = scaler.fit_transform(X)
    km = KMeans(
        n_clusters=N_CLUSTERS,
        n_init=10,
        random_state=RANDOM_STATE
    )
    labels = km.fit_predict(Xz)
else:
    # too few edges or a single cluster: all edges go to cluster 0
    labels = np.zeros((len(edge_tuples),), dtype=int)
    N_CLUSTERS = 1  # keeps the later shapes consistent

# edge_cluster of each edge
df_edge_clusters = pd.DataFrame({
    "tf_idx": [e[0] for e in edge_tuples],
    "tg_idx": [e[1] for e in edge_tuples],
    "tf_name": [tf_idx2name[e[0]] for e in edge_tuples],
    "tg_name": [tg_idx2name[e[1]] for e in edge_tuples],
    "edge_cluster": labels
})
df_edge_clusters.to_csv(
    os.path.join(OUTPUT_DIR,
                 f"edge_clusters_{N_CLUSTERS}k_from_network_{SELECTION_MODE}.csv"),
    index=False
)

# 7) counts and fractions of each edge_cluster per cell_cluster
counts = np.zeros((N_CLUSTERS, len(CLUSTERS)), dtype=np.float32)
for (row_idx, lab) in enumerate(labels):
    counts[lab, :] += X[row_idx, :]

tot_per_ec = counts.sum(axis=1, keepdims=True)
with np.errstate(invalid="ignore", divide="ignore"):
    prop = counts / np.maximum(tot_per_ec, 1.0)

df_counts = pd.DataFrame(counts, columns=[f"C{cid}" for cid in CLUSTERS])
df_counts.insert(0, "edge_cluster", np.arange(N_CLUSTERS))
df_prop = pd.DataFrame(prop, columns=[f"C{cid}" for cid in CLUSTERS])
df_prop.insert(0, "edge_cluster", np.arange(N_CLUSTERS))

df_counts.to_csv(
    os.path.join(OUTPUT_DIR,
                 f"edgeCluster_x_cellCluster_counts_from_network_{SELECTION_MODE}.csv"),
    index=False
)
df_prop.to_csv(
    os.path.join(OUTPUT_DIR,
                 f"edgeCluster_x_cellCluster_prop_from_network_{SELECTION_MODE}.csv"),
    index=False
)

# ================= global font settings =================
plt.rcParams.update({
    "axes.titlesize": 28,   # title
    "axes.labelsize": 20,   # axis labels
    "xtick.labelsize": 20,  # x ticks
    "ytick.labelsize": 20,  # y ticks
    "font.size": 20         # base size (legends, text)
})
# ======================================================

# 8) heatmap (edge_cluster × cell_cluster); the bars give each edge_cluster's edge count
ec_edge_counts = counts.sum(axis=1)  # shape [N_CLUSTERS]
xt = [f"C{cid}" for cid in CLUSTERS]
yt = [f"EC{k}" for k in range(N_CLUSTERS)]

# line break in the title keeps large fonts inside the figure
plot_heatmap_with_ec_bar(
    counts, xt, yt,
    f"Edge Clustering",
    f"heatmap_counts_{N_CLUSTERS}k_from_network_{SELECTION_MODE}.png",
    ec_edge_counts
)

plot_heatmap_with_ec_bar(
    prop, xt, yt,
    f"Edge Clustering",
    f"heatmap_prop_{N_CLUSTERS}k_from_network_{SELECTION_MODE}.png",
    ec_edge_counts
)

print(f"Done. Edges used: {len(df_edges_sorted)}; "
      f"union edges: {len(edge_tuples)}; "
      f"edge clusters: {N_CLUSTERS}.")
print(f"Outputs saved in: {OUTPUT_DIR}")

In [ ]:
# === Edge-cluster Top-5 TF dotplot ===
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# 1) attach edge_cluster to the edge table (color = mean score)
#    df_edges can hold the same (tf, tg) in several cell clusters; colors use the selected edges after the merge
edge_with_ec = pd.merge(
    df_edges[["tf_idx","tg_idx","tf_name","tg_name","score"]],
    df_edge_clusters[["tf_idx","tg_idx","edge_cluster"]],
    on=["tf_idx","tg_idx"],
    how="left"
)

# 2) TF degree (unique TGs) and color value (mean score) within each edge_cluster
#    degree counts unique TGs (not occurrences); color is the mean score of the TF's selected edges
deg_per_ec_tf = (df_edge_clusters
                 .groupby(["edge_cluster","tf_idx","tf_name"])["tg_idx"]
                 .nunique()
                 .reset_index(name="degree"))
color_per_ec_tf = (edge_with_ec
                   .groupby(["edge_cluster","tf_idx","tf_name"])["score"]
                   .mean()
                   .reset_index(name="mean_score"))

stats_tf = pd.merge(deg_per_ec_tf, color_per_ec_tf,
                    on=["edge_cluster","tf_idx","tf_name"], how="left")

# 3) top-5 TFs per edge_cluster (by descending degree; ties by name)
stats_tf["rank_in_ec"] = stats_tf.groupby("edge_cluster")["degree"] \
                                 .rank(method="first", ascending=False)
topk = stats_tf[stats_tf["rank_in_ec"] <= 10].copy()

# skip when there is no data
if topk.empty:
    print("No TFs to plot for edge-cluster Top-5; skip dotplot.")
else:
    # 4) bubble-plot data: x = edge_cluster, y = TF, size = degree, color = mean_score
    #    y order: total selected degree of each TF (summed over ECs), high to low
    tf_order = (topk.groupby(["tf_idx","tf_name"])["degree"]
                     .sum().sort_values(ascending=False)
                     .reset_index()["tf_name"].tolist())
    topk["tf_name"] = pd.Categorical(topk["tf_name"], categories=tf_order, ordered=True)
    topk.sort_values(["tf_name","edge_cluster"], inplace=True)

    # x axis: EC0..EC{N-1}
    ec_labels = [f"EC{k}" for k in sorted(topk["edge_cluster"].unique())]
    xmap = {lab:i for i, lab in enumerate(ec_labels)}
    topk["EC"] = topk["edge_cluster"].map(lambda k: f"EC{k}")

    # point sizes scaled linearly to 50-500
    max_deg = int(max(1, topk["degree"].max()))
    topk["size"] = 50 + 450 * (topk["degree"] / max_deg)

    # colors: HEATMAP_CMAP as in the heatmap; NaN in gray; scores usually 0.5-1.0
    cmap = mpl.cm.get_cmap(HEATMAP_CMAP).copy()
    cmap.set_bad((0.85, 0.85, 0.85, 1.0))
    norm = mpl.colors.Normalize(vmin=0.5, vmax=1.0)
    vals_masked = np.ma.masked_invalid(topk["mean_score"].to_numpy())

    # 5) plot (scatter as the mappable; same colors and style as above)
    fig_h = max(6, 0.35*len(tf_order))
    fig_w = max(8, 0.7*len(ec_labels))
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sc = ax.scatter(
        [xmap[x] for x in topk["EC"]],
        [tf_order.index(y) for y in topk["tf_name"].astype(str)],
        s=topk["size"].to_numpy(),
        c=vals_masked,
        cmap=cmap,
        norm=norm,
        alpha=0.9, edgecolors="k", linewidths=0.3
    )

    ax.set_xticks(list(range(len(ec_labels))))
    ax.set_xticklabels(ec_labels, rotation=45, ha="right")
    ax.set_yticks(list(range(len(tf_order))))
    ax.set_yticklabels(tf_order)
    ax.set_title(f"Top-5 TF per edge-cluster (size=degree, color=mean score) — {SELECTION_MODE}")

    cb = fig.colorbar(sc, ax=ax)
    cb.set_label("mean(score)")

    fig.tight_layout()
    if SAVE_FIG:
        fig.savefig(os.path.join(OUTPUT_DIR, f"ec_top5TF_dotplot_{SELECTION_MODE}.png"), dpi=220)
    plt.show()

    # export the plotted data
    out_cols = ["edge_cluster","tf_idx","tf_name","degree","mean_score"]
    topk[out_cols].sort_values(["edge_cluster","degree"], ascending=[True, False]) \
                   .to_csv(os.path.join(OUTPUT_DIR, f"ec_top5TF_dotplot_data_{SELECTION_MODE}.csv"),
                           index=False)


In [ ]:
import math
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# =========================
# 0) font sizes
# =========================
FS_YTICK = 11
FS_XLABEL = 11
FS_TITLE = 12

# =========================
# 1) activity prop of each EC per cell_cluster, and the title strings
#    prop = edges of the EC present in the cluster / all edges of the EC
# =========================
THR_PROP = 0.10  # 10% threshold

# df_edges needs cell_cluster; deduplicate (edge, cluster) to avoid double counting
edge_occ = df_edges[["tf_idx", "tg_idx", "cell_cluster"]].drop_duplicates()

# attach edge_cluster to each (edge, cluster)
edge_occ = edge_occ.merge(
    df_edge_clusters[["tf_idx", "tg_idx", "edge_cluster"]],
    on=["tf_idx", "tg_idx"],
    how="left"
)

# edge_count of each EC in each cluster
ec_cc_counts = (edge_occ
                .groupby(["edge_cluster", "cell_cluster"])
                .size()
                .reset_index(name="edge_count"))

# total (unique) edges of each EC
ec_total = (df_edge_clusters
            .groupby("edge_cluster")
            .size()
            .reset_index(name="total_edges"))

ec_cc = ec_cc_counts.merge(ec_total, on="edge_cluster", how="left")
ec_cc["prop"] = ec_cc["edge_count"] / np.maximum(ec_cc["total_edges"], 1)

# dict ec -> title suffix such as "C4(0.35), C5(0.18)"
ec2title = {}
for ec in sorted(ec_total["edge_cluster"].unique()):
    sub = ec_cc[ec_cc["edge_cluster"] == ec].copy()
    if sub.empty:
        ec2title[ec] = "NA"
        continue

    sub.sort_values("prop", ascending=False, inplace=True)

    sub_thr = sub[sub["prop"] >= THR_PROP].copy()
    if sub_thr.empty:
        sub_thr = sub.head(1)      # if none reaches 10%, show the top one
    else:
        sub_thr = sub_thr.head(3)  # at most 3 clusters

    parts = [f"C{int(r.cell_cluster)}({r.prop:.2f})" for r in sub_thr.itertuples(index=False)]
    ec2title[ec] = ", ".join(parts)

# =========================
# 2) TF centrality and mean_score
# =========================
edge_with_ec = pd.merge(
    df_edges[["tf_idx","tg_idx","tf_name","tg_name","score"]],
    df_edge_clusters[["tf_idx","tg_idx","edge_cluster"]],
    on=["tf_idx","tg_idx"],
    how="left"
)

deg_per_ec_tf = (df_edge_clusters
                 .groupby(["edge_cluster","tf_idx","tf_name"])["tg_idx"]
                 .nunique()
                 .reset_index(name="degree"))

color_per_ec_tf = (edge_with_ec
                   .groupby(["edge_cluster","tf_idx","tf_name"])["score"]
                   .mean()
                   .reset_index(name="mean_score"))

stats_tf = pd.merge(deg_per_ec_tf, color_per_ec_tf,
                    on=["edge_cluster","tf_idx","tf_name"], how="left")

# =========================
# 3) one figure, 3 subplots per row
# =========================
TOPK_PER_EC = 12
HEATMAP_CMAP = HEATMAP_CMAP  # same colormap as above
cmap = mpl.cm.get_cmap(HEATMAP_CMAP).copy()
cmap.set_bad((0.85, 0.85, 0.85, 1.0))
norm = mpl.colors.Normalize(vmin=0.5, vmax=1.0)  # color range of the scores

ecs = sorted(stats_tf["edge_cluster"].dropna().unique().tolist())
n_ec = len(ecs)
ncols = 3
nrows = math.ceil(n_ec / ncols)

# about 4.6 × 3.2 per subplot
fig_w = 4.6 * ncols
fig_h = 4.2 * nrows
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(fig_w, fig_h))
axes = np.array(axes).reshape(-1)

for i, ec in enumerate(ecs):
    ax = axes[i]

    sub = stats_tf[stats_tf["edge_cluster"] == ec].copy()
    if sub.empty:
        ax.set_axis_off()
        continue

    sub.sort_values("degree", ascending=False, inplace=True)
    sub_top = sub.head(TOPK_PER_EC).copy()

    # ascending order puts the largest bar on top (barh draws bottom-up)
    sub_plot = sub_top.sort_values("degree", ascending=True)

    y_labels = sub_plot["tf_name"].tolist()
    y_pos = np.arange(len(sub_plot))
    bars = sub_plot["degree"].to_numpy()
    colors_vals = sub_plot["mean_score"].to_numpy()

    colors = cmap(norm(np.nan_to_num(colors_vals, nan=0.0)))
    isnan = np.isnan(colors_vals)
    colors[isnan] = (0.85, 0.85, 0.85, 1.0)

    ax.barh(y_pos, bars, color=colors, edgecolor="k", linewidth=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=FS_YTICK)

    ax.set_xlabel("centrality (unique TG count)", fontsize=FS_XLABEL)

    title_suffix = ec2title.get(ec, "NA")
    ax.set_title(f"EC{ec} | {title_suffix}", fontsize=FS_TITLE)

    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.tick_params(axis="x", labelsize=FS_YTICK)

# hide unused subplots
for j in range(n_ec, len(axes)):
    axes[j].set_axis_off()

# spacing for titles and labels; left margin for the y labels
fig.subplots_adjust(left=0.08, right=0.98, top=0.92, bottom=0.08, wspace=0.45, hspace=0.65)

if SAVE_FIG:
    out_png = os.path.join(OUTPUT_DIR, f"all_ec_top{TOPK_PER_EC}_TF_centrality_grid3_{SELECTION_MODE}.png")
    fig.savefig(out_png, dpi=220)

plt.show()

# =========================
# 4) summary CSV
# =========================
summary = (stats_tf.sort_values(["edge_cluster","degree"], ascending=[True, False])
                    .groupby("edge_cluster").head(TOPK_PER_EC))

summary[["edge_cluster","tf_idx","tf_name","degree","mean_score"]].to_csv(
    os.path.join(OUTPUT_DIR, f"ec_top{TOPK_PER_EC}_TF_centrality_bar_data_{SELECTION_MODE}.csv"),
    index=False
)

In [ ]:
# === Module-specific TFs: top TFs of each edge_cluster by specificity S_{k,t}, as a bubble plot ===

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap

# parameters
TOPK_TF_PER_EC_SPEC = 20       # specific TFs kept per edge_cluster
MIN_DEG_IN_MODULE    = 2      # minimum TGs in the module for a TF to count (removes noise TFs)

# 1) attach edge_cluster to the edge table: edge_with_ec
#    (recomputed here)
edge_with_ec = df_edges.merge(
    df_edge_clusters[["tf_idx", "tg_idx", "edge_cluster"]],
    on=["tf_idx", "tg_idx"],
    how="inner"
)

# 2) deg_{k,t}: centrality of the TF in each module (unique TGs)
deg_kt = (
    edge_with_ec
    .groupby(["edge_cluster", "tf_idx", "tf_name"])["tg_idx"]
    .nunique()                       # unique TGs
    .reset_index(name="deg_kt")
)

# 3) total activity of each TF over all modules, deg_all_t
deg_all = (
    deg_kt
    .groupby(["tf_idx", "tf_name"])["deg_kt"]
    .sum()
    .reset_index(name="deg_all_t")
)

deg_kt = deg_kt.merge(deg_all, on=["tf_idx", "tf_name"], how="left")

# 4) maximum centrality within each module, max_deg_k, to normalize c_{k,t}
max_deg_k = (
    deg_kt
    .groupby("edge_cluster")["deg_kt"]
    .max()
    .reset_index(name="max_deg_k")
)

deg_kt = deg_kt.merge(max_deg_k, on="edge_cluster", how="left")

# 5) compute:
#    p_{k,t} = deg_{k,t} / deg_all_t     (share of the TF's activity in this module)
#    c_{k,t} = deg_{k,t} / max_deg_k     (how central the TF is within the module)
#    S_{k,t} = p_{k,t} * c_{k,t}         (module specificity × importance within the module)
eps = 1e-8
deg_kt["p_kt"] = deg_kt["deg_kt"] / (deg_kt["deg_all_t"] + eps)
deg_kt["c_kt"] = deg_kt["deg_kt"] / (deg_kt["max_deg_k"] + eps)
deg_kt["S_kt"] = deg_kt["p_kt"] * deg_kt["c_kt"]

# 6) drop TFs with few TGs in the module
deg_kt_filt = deg_kt.query("deg_kt >= @MIN_DEG_IN_MODULE").copy()

# 7) top TOPK_TF_PER_EC_SPEC TFs by S_{k,t} per edge_cluster
top_spec = (
    deg_kt_filt
    .sort_values(["edge_cluster", "S_kt"], ascending=[True, False])
    .groupby("edge_cluster")
    .head(TOPK_TF_PER_EC_SPEC)
    .reset_index(drop=True)
)

# report modules without TFs after filtering
missing_ec = sorted(set(deg_kt["edge_cluster"]) - set(top_spec["edge_cluster"]))
if missing_ec:
    print(f"[Info] Modules without specific TFs at this threshold: {missing_ec}")

# save the table
out_csv = os.path.join(OUTPUT_DIR, f"edgeCluster_top{TOPK_TF_PER_EC_SPEC}_TF_module_specificity.csv")
top_spec.to_csv(out_csv, index=False)
print(f"Saved module-specific TF table to: {out_csv}")

# 8) bubble-plot data

# 8.1 global TF order (by S_kt)
tf_order_df = (
    top_spec
    .groupby(["tf_idx", "tf_name"])["S_kt"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

tf_labels = tf_order_df["tf_name"].tolist()
tf2y = {name: i for i, name in enumerate(tf_labels)}

# 8.2 x axis: edge_cluster (by number)
ec_ids = sorted(top_spec["edge_cluster"].unique())
x_labels = [f"EC{ec}" for ec in ec_ids]
ec2x = {ec: i for i, ec in enumerate(ec_ids)}

# 8.3 point positions, sizes and colors
xs = top_spec["edge_cluster"].map(ec2x).to_numpy()
ys = top_spec["tf_name"].map(tf2y).to_numpy()

# bubble size: deg_kt (TGs in the module), scaled linearly
deg_vals = top_spec["deg_kt"].to_numpy().astype(float)
if deg_vals.max() > deg_vals.min():
    size_min, size_max = 80.0, 600.0
    sizes = size_min + (deg_vals - deg_vals.min()) / (deg_vals.max() - deg_vals.min()) * (size_max - size_min)
else:
    sizes = np.full_like(deg_vals, 300.0)

# color: module specificity S_kt
S_vals = top_spec["S_kt"].to_numpy().astype(float)
cmap = get_cmap(HEATMAP_CMAP if "HEATMAP_CMAP" in globals() else "OrRd")
norm = Normalize(vmin=S_vals.min(), vmax=S_vals.max())

# 9) bubble plot
fig_w = max(6, 0.7 * len(ec_ids))
fig_h = max(4, 0.45 * len(tf_labels))
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sc = ax.scatter(
    xs,
    ys,
    s=sizes,
    c=S_vals,
    cmap=cmap,
    norm=norm,
    edgecolors="k",
    linewidths=0.5,
    alpha=0.85,
)

ax.set_xticks(np.arange(len(x_labels)))
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.set_yticks(np.arange(len(tf_labels)))
ax.set_yticklabels(tf_labels)

ax.set_xlabel("Edge cluster")
ax.set_ylabel("TF")
ax.set_title(
    f"Top-{TOPK_TF_PER_EC_SPEC} module-specific TF per edge_cluster\n"
    r"(bubble size = deg$_{k,t}$, color = $S_{k,t}$)"
)

cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"TF module-specificity $S_{k,t}$")

# first TFs at the top
ax.invert_yaxis()

ax.grid(axis="x", linestyle="--", alpha=0.2)
fig.tight_layout()

if "SAVE_FIG" in globals() and SAVE_FIG:
    out_fig = os.path.join(OUTPUT_DIR, f"edgeCluster_top{TOPK_TF_PER_EC_SPEC}_TF_module_specificity_bubble.png")
    fig.savefig(out_fig, dpi=220)
    print(f"Saved bubble plot to: {out_fig}")

plt.show()


In [ ]:
# === Bar charts of the specific TFs of each edge_cluster ===

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap

# uses top_spec / ec_ids from the previous cell
# top_spec columns include edge_cluster, tf_name, deg_kt, p_kt, c_kt, S_kt

if 'top_spec' not in globals():
    raise RuntimeError("Run the previous cell first to compute top_spec (the specific TFs of each module).")

# common color range (S_kt over all modules)
S_all = top_spec["S_kt"].to_numpy().astype(float)
S_min, S_max = float(S_all.min()), float(S_all.max())
cmap = get_cmap(HEATMAP_CMAP if "HEATMAP_CMAP" in globals() else "OrRd")
norm = Normalize(vmin=S_min, vmax=S_max)

# edge_cluster list
ec_ids = sorted(top_spec["edge_cluster"].unique())

for ec in ec_ids:
    sub = top_spec[top_spec["edge_cluster"] == ec].copy()
    if sub.empty:
        continue

    # ascending S_kt, so specificity increases upwards
    sub = sub.sort_values("S_kt", ascending=True)

    tf_names = sub["tf_name"].tolist()
    S_vals   = sub["S_kt"].to_numpy().astype(float)
    deg_vals = sub["deg_kt"].to_numpy().astype(float)

    y_pos = np.arange(len(sub))
    colors = cmap(norm(S_vals))

    fig_h = max(3.5, 0.4 * len(sub))
    fig_w = 6
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    bars = ax.barh(
        y_pos,
        S_vals,
        color=colors,
        edgecolor="k",
        alpha=0.9,
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(tf_names)
    ax.set_xlabel(r"Module-specificity $S_{k,t}$")
    ax.set_title(
        f"EC{ec}: Top-{len(sub)} module-specific TF\n"
        r"(height = $S_{k,t}$, number of TG shown on the right)"
    )

    # label each bar with the TF's TG count in the module (deg_kt)
    for y, s, d in zip(y_pos, S_vals, deg_vals):
        ax.text(
            s * 1.01,
            y,
            f"{int(d)} TG",
            va="center",
            ha="left",
            fontsize=9,
        )

    # colorbar (S_kt)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # required by the colorbar
    cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label(r"$S_{k,t}$")

    fig.tight_layout()

    if "SAVE_FIG" in globals() and SAVE_FIG:
        out_fig = os.path.join(
            OUTPUT_DIR,
            f"edgeCluster_EC{ec}_topTF_module_specificity_bar.png"
        )
        fig.savefig(out_fig, dpi=220)
        print(f"Saved bar plot for EC{ec} to: {out_fig}")

    plt.show()
